# Pandas from the ground up: UN General Debate speeches and economic data

This notebook teaches pandas by building one real project step by step: a text analysis of the
UN General Debate Corpus (one speech per country per year, 1970 onwards), merged with a World Bank
economic variable (GDP per capita), ending with a regression in the style of your econometrics course.

Every chapter builds on the previous one. Run the cells in order the first time. Later you can jump
around, but the variable names assume the earlier chapters have run.

**Chapters**

0. How to use this notebook
1. The Python you need before pandas
2. Series and DataFrame: the two objects everything is built on
3. Loading the corpus
4. Looking at data: selecting, filtering, sorting
5. Cleaning
6. New columns and basic text measures
7. Keyword counting: the core text method
8. groupby: from speeches to countries and years
9. Reshaping: wide and long
10. The economic variable: World Bank data and merge
11. Time series tools
12. Plots
13. First statistics
14. Word frequencies and distinctive words
15. From notebook to project

Each chapter ends with exercises. Solutions are hidden behind a "Solution" toggle. Try first, then look.

## Chapter 0: How to use this notebook

**Jupyter basics**

- A notebook is a list of cells. A cell is either Markdown (text, like this one) or Code.
- Click a cell and press `Shift+Enter` to run it and move to the next one. `Ctrl+Enter` runs it and stays.
- The number in brackets on the left, like `[3]`, is the execution order. A `[*]` means it is still running.
- Variables live in the kernel's memory. If you restart the kernel (Kernel menu), everything is forgotten
  and you must run the cells again from the top. Do this whenever things behave strangely.
- `Esc` then `a` inserts a cell above, `b` below, `dd` deletes, `m` turns a cell into Markdown, `y` into Code.
- The last expression in a code cell is displayed automatically. Everything else needs `print()`.

**Getting help inside the notebook**

- `df.head?` shows the documentation of a method. `df.head??` shows its source code.
- Type `df.` and press `Tab` to see every method available.
- `help(pd.read_csv)` is the long form.

**Project folder**

This notebook expects the following structure next to it. Create the folders now if they do not exist
(the next cell does it for you).

```
project/
    pandas_un_speeches.ipynb     <- this notebook
    data/
        raw/                     <- original downloads, never edited
        processed/               <- files this notebook creates
    figures/                     <- plots you want to keep
```

**The data**

1. UN General Debate Corpus. The easiest version is the Kaggle CSV `un-general-debates.csv`
   (dataset "unitednations/un-general-debates"). Put it in `data/raw/`. The original and most current version
   lives on Harvard Dataverse (doi:10.7910/DVN/0TJX8Y) as a zip of text files; Chapter 3 shows how to load that too.
2. World Bank GDP per capita. Chapter 10 downloads it with the `wbgapi` package, or you download the CSV
   from data.worldbank.org by hand.

If neither file is present, the notebook builds a small synthetic corpus so that every cell still runs.
That is useful for learning the mechanics, but swap in the real data as soon as you have it.

In [ ]:
# Imports used throughout the notebook. Run this first, and again after every kernel restart.
import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create the folder structure described above (does nothing if it already exists)
for folder in ["data/raw", "data/processed", "figures"]:
    os.makedirs(folder, exist_ok=True)

# A few display settings so wide tables are readable
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 80)

print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)

If the cell above fails with `ModuleNotFoundError`, install the missing package from a terminal:

```
pip install pandas numpy matplotlib
```

Later chapters additionally use `statsmodels`, `scikit-learn`, `wbgapi` and `pyarrow`. Install them all at once:

```
pip install statsmodels scikit-learn wbgapi pyarrow
```

## Chapter 1: The Python you need before pandas

Pandas is a library written in Python, and its methods take Python objects as arguments. You need
a working command of six things: variables, lists, dictionaries, strings, loops and functions. This chapter
is a compressed refresher. If all of it is familiar, do the exercises and move on.

### 1.1 Variables and basic types

Python has no type declarations. A variable is a name pointing at a value, and the value has a type.

In [ ]:
year = 1995            # int
share = 0.42           # float
country = "CHE"        # str
is_rich = True         # bool

print(type(year), type(share), type(country), type(is_rich))

# Arithmetic works as expected. Division always gives a float, // gives an integer.
print(7 / 2, 7 // 2, 7 % 2, 2 ** 10)

### 1.2 Lists

A list is an ordered collection. Indexing starts at 0. Negative indices count from the end.
Slicing `a[start:stop]` includes `start` and excludes `stop`. This same convention is used by pandas `iloc`.

In [ ]:
codes = ["CHE", "DEU", "FRA", "USA", "CHN"]

print(codes[0])        # first element
print(codes[-1])       # last element
print(codes[1:3])      # elements 1 and 2, not 3
print(len(codes))      # number of elements

codes.append("IND")    # add at the end (modifies the list in place)
print(codes)

# A list can hold anything, including other lists
rows = [["CHE", 1995, 0.42], ["DEU", 1995, 0.37]]
print(rows[1][0])

### 1.3 Dictionaries

A dictionary maps keys to values. It is the most important structure for pandas, because a DataFrame is
conceptually a dictionary of columns, and `pd.DataFrame({...})` takes a dictionary.

In [ ]:
gdp = {"CHE": 45000, "DEU": 31000, "FRA": 28000}

print(gdp["CHE"])            # look up a key
gdp["USA"] = 29000           # add a key
print(gdp.keys())
print(gdp.values())
print("CHE" in gdp)          # membership test

# .get returns a default instead of an error when the key is missing
print(gdp.get("IND", "not available"))

### 1.4 Strings

Speeches are strings, so string methods matter. Everything here has a pandas counterpart under `.str`
that applies the same operation to a whole column at once (Chapter 5).

In [ ]:
s = "  The General Assembly discussed Trade, trade and TRADE.  "

print(s.strip())                 # remove whitespace at both ends
print(s.lower())                 # all lowercase
print(s.lower().count("trade"))  # count occurrences
print(s.split())                 # split on whitespace -> list of words
print(len(s.split()))            # word count, crude but useful
print(s.replace("Trade", "commerce"))
print("trade" in s.lower())      # substring test

# f-strings put variables into text. Use them for printing results.
n = len(s.split())
print(f"The sentence has {n} words and {n/3:.1f} words per third.")

### 1.5 Loops and conditions

You will use loops less than you expect, because pandas applies operations to whole columns at once.
But loops are needed for reading many files, iterating over keywords, and building lists.

In [ ]:
keywords = ["trade", "inflation", "debt"]
text = "Debt relief and fair trade are the priorities. Trade must be free."

counts = {}
for kw in keywords:
    counts[kw] = text.lower().count(kw)
print(counts)

# Conditions
for kw, n in counts.items():
    if n == 0:
        print(kw, "is not mentioned")
    elif n == 1:
        print(kw, "is mentioned once")
    else:
        print(kw, "is mentioned", n, "times")

# List comprehension: a loop that builds a list in one line. Very common in pandas code.
lengths = [len(kw) for kw in keywords]
print(lengths)

long_keywords = [kw for kw in keywords if len(kw) > 4]
print(long_keywords)

### 1.6 Functions

Wrap any logic you use twice in a function. Later (Chapter 15) functions move into a separate `.py` file.

In [ ]:
def count_keywords(text, keywords):
    """Return a dictionary with the number of occurrences of each keyword in text (case-insensitive)."""
    text_lower = text.lower()
    return {kw: text_lower.count(kw) for kw in keywords}

print(count_keywords("Trade, trade, and more trade. Also debt.", ["trade", "debt", "war"]))

# Default arguments and keyword arguments
def per_thousand(count, n_words, digits=2):
    return round(1000 * count / n_words, digits)

print(per_thousand(7, 2500))
print(per_thousand(7, 2500, digits=4))

### Exercises 1

1. Write a function `word_count(text)` that returns the number of words in a string.
2. Given `speeches = ["Peace and trade.", "Debt, debt, debt.", "Climate action now."]`, build a list with the
   word count of each speech using a list comprehension.
3. Build a dictionary that maps each keyword in `["peace", "debt", "climate"]` to the number of speeches
   in the list above that mention it.

<details>
<summary>Solution</summary>

```python
def word_count(text):
    return len(text.split())

speeches = ["Peace and trade.", "Debt, debt, debt.", "Climate action now."]
wc = [word_count(s) for s in speeches]

mentions = {}
for kw in ["peace", "debt", "climate"]:
    mentions[kw] = sum(1 for s in speeches if kw in s.lower())
print(wc, mentions)
```
</details>

In [ ]:
# Your solution to Exercises 1

## Chapter 2: Series and DataFrame

Pandas has two central objects.

- A **Series** is one column: a one-dimensional array of values with an **index** (labels for each row).
- A **DataFrame** is a table: several Series that share the same index.

Everything else in pandas is a method on one of these two. When something is confusing, ask yourself:
is this a Series or a DataFrame, and what is its index?

In [ ]:
# A Series from a list. The index defaults to 0, 1, 2, ...
s = pd.Series([45000, 31000, 28000, 29000])
s

In [ ]:
# A Series with a meaningful index
gdp = pd.Series([45000, 31000, 28000, 29000], index=["CHE", "DEU", "FRA", "USA"], name="gdp_pc")
gdp

In [ ]:
print(gdp["DEU"])       # label-based lookup, like a dictionary
print(gdp.index)
print(gdp.values)       # the underlying numpy array
print(gdp.mean(), gdp.max(), gdp.idxmax())   # idxmax returns the label of the maximum

Operations on a Series apply to every element at once. This is called **vectorization** and it is the single
most important habit to build: whenever you are about to write a `for` loop over rows, check whether a
vectorized operation does the same thing.

In [ ]:
print(gdp / 1000)          # divide every element
print(gdp > 30000)         # compare every element -> a Series of booleans
print(np.log(gdp))         # numpy functions work elementwise too

### 2.2 DataFrame

The most common way to build a small DataFrame by hand is from a dictionary of lists: keys become column
names, lists become columns.

In [ ]:
df = pd.DataFrame({
    "country": ["CHE", "DEU", "FRA", "USA", "CHN"],
    "year":    [1995, 1995, 1995, 1995, 1995],
    "gdp_pc":  [45000, 31000, 28000, 29000, 600],
    "text":    ["Peace and trade.", "Debt, debt, debt.", "Climate action now.",
                "Trade and security.", "Development first."],
})
df

In [ ]:
# The first things to check on any DataFrame
print(df.shape)        # (rows, columns)
print(df.columns)
print(df.dtypes)       # the type of each column. 'object' or 'str' means text.
print(df.index)

In [ ]:
df.info()     # types, non-missing counts, memory. Read this for every new dataset.

In [ ]:
df.describe()     # summary statistics of the numeric columns

In [ ]:
# Selecting one column gives a Series
df["gdp_pc"]

In [ ]:
# Selecting a list of columns gives a DataFrame. Note the double brackets.
df[["country", "gdp_pc"]]

In [ ]:
# Adding a column: assign a Series or a list of the right length
df["log_gdp"] = np.log(df["gdp_pc"])
df["n_words"] = df["text"].str.split().str.len()   # .str applies string methods to every element
df

### 2.3 The index

The index is the row labels. By default it is 0..n-1, which is fine most of the time. You can set a column
as index with `set_index` and go back with `reset_index`. Many operations (merging, aligning, plotting)
behave differently depending on the index, so always know what it is.

In [ ]:
df_indexed = df.set_index("country")
df_indexed

In [ ]:
print(df_indexed.loc["CHE"])            # row by label
print(df_indexed.loc["CHE", "gdp_pc"])  # single value
df_indexed.reset_index().head(2)        # back to a default index; 'country' becomes a column again

### 2.4 Methods return new objects

Almost every pandas method returns a new object and leaves the original untouched. You must assign the
result to keep it. This is a frequent source of "my change disappeared" confusion.

In [ ]:
df.sort_values("gdp_pc")      # returns a sorted copy and displays it
print(df["country"].tolist()) # the original is unchanged

df = df.sort_values("gdp_pc") # now it is kept
print(df["country"].tolist())

### Exercises 2

1. Build a DataFrame with three countries, two years each (six rows), and a `gdp_pc` column.
2. Add a column `gdp_pc_thousands`.
3. Print the mean GDP per capita and the country with the highest value.
4. Set `country` as index, look up one row with `.loc`, and reset the index again.

<details>
<summary>Solution</summary>

```python
ex = pd.DataFrame({
    "country": ["CHE", "CHE", "DEU", "DEU", "IND", "IND"],
    "year": [2000, 2010, 2000, 2010, 2000, 2010],
    "gdp_pc": [38000, 74000, 23000, 41000, 440, 1350],
})
ex["gdp_pc_thousands"] = ex["gdp_pc"] / 1000
print(ex["gdp_pc"].mean())
print(ex.loc[ex["gdp_pc"].idxmax(), "country"])
exi = ex.set_index("country")
print(exi.loc["IND"])
ex = exi.reset_index()
```
</details>

In [ ]:
# Your solution to Exercises 2

## Chapter 3: Loading the corpus

The goal of this chapter is one tidy DataFrame named `speeches` with exactly these columns:

| column    | type | meaning                                  |
|-----------|------|------------------------------------------|
| `country` | str  | ISO3 code of the speaking state (e.g. CHE) |
| `session` | int  | UN General Assembly session number        |
| `year`    | int  | calendar year                             |
| `text`    | str  | full speech text                          |

"Tidy" means: one row per observation (here: one speech), one column per variable, one value per cell.
Every later chapter assumes this shape. Getting data into this shape is usually half the work of a project.

There are three ways to get there, depending on which version of the data you have. Only one of them
needs to run.

### 3.1 Option A: the Kaggle CSV

`pd.read_csv` is the workhorse. It reads a CSV into a DataFrame and guesses the types. The Kaggle file
already has the four columns we want.

In [ ]:
KAGGLE_CSV = "data/raw/un-general-debates.csv"

speeches = None
SYNTHETIC = False      # becomes True only if we fall back to the practice corpus
if os.path.exists(KAGGLE_CSV):
    speeches = pd.read_csv(KAGGLE_CSV)
    print("Loaded Kaggle CSV:", speeches.shape)
else:
    print("Kaggle CSV not found, trying the next option.")

### 3.2 Option B: the Dataverse text files

The Dataverse version is a zip with one folder per session and one `.txt` file per speech, named like
`CHE_25_1970.txt`. There is no table; we must build it. This is a very common situation and worth
learning: loop over files, pull metadata out of the filename with a regular expression, collect rows in
a list of dictionaries, and turn the list into a DataFrame at the end.

Unzip the archive into `data/raw/ungdc/` first.

In [ ]:
DATAVERSE_DIR = "data/raw/ungdc"

def load_from_txt_files(root):
    """Build the speeches table from the Dataverse folder of .txt files."""
    pattern = re.compile(r"([A-Z]{2,4})_(\d+)_(\d{4})\.txt$")
    rows = []
    # recursive=True with ** walks into all subfolders
    for path in glob.glob(os.path.join(root, "**", "*.txt"), recursive=True):
        m = pattern.search(os.path.basename(path))
        if m is None:
            continue
        with open(path, encoding="utf-8", errors="replace") as f:
            text = f.read()
        rows.append({
            "country": m.group(1),
            "session": int(m.group(2)),
            "year": int(m.group(3)),
            "text": text,
        })
    return pd.DataFrame(rows)

if speeches is None and os.path.isdir(DATAVERSE_DIR):
    speeches = load_from_txt_files(DATAVERSE_DIR)
    print("Loaded Dataverse text files:", speeches.shape)

### 3.3 Option C: a synthetic corpus for practice

If you have no data yet, the next cell generates a small artificial corpus with the same four columns.
The texts are built from a pool of sentences, with keyword frequencies that depend on a hidden income level
and on the year, so the later analysis produces recognizable patterns. You do not need to understand this
cell. It also plants a few typical data problems (duplicates, stray characters, messy whitespace) so that
the cleaning chapter has something to clean.

In [ ]:
def make_synthetic_corpus(seed=42):
    rng = np.random.default_rng(seed)
    # (ISO3, region, base GDP per capita in 1970 USD)
    countries = [
        ("CHE", "Europe", 4000), ("DEU", "Europe", 2800), ("FRA", "Europe", 2900), ("GBR", "Europe", 2300),
        ("ITA", "Europe", 2100), ("SWE", "Europe", 4200), ("POL", "Europe", 900), ("GRC", "Europe", 1300),
        ("USA", "Americas", 5200), ("CAN", "Americas", 4100), ("MEX", "Americas", 700), ("BRA", "Americas", 450),
        ("ARG", "Americas", 1300), ("CHL", "Americas", 900), ("COL", "Americas", 350), ("PER", "Americas", 500),
        ("CHN", "Asia", 110), ("IND", "Asia", 110), ("JPN", "Asia", 2000), ("KOR", "Asia", 280),
        ("IDN", "Asia", 80), ("PAK", "Asia", 170), ("THA", "Asia", 190), ("PHL", "Asia", 190),
        ("NGA", "Africa", 220), ("KEN", "Africa", 140), ("EGY", "Africa", 230), ("ZAF", "Africa", 800),
        ("GHA", "Africa", 250), ("ETH", "Africa", 80), ("TZA", "Africa", 100), ("MAR", "Africa", 260),
        ("AUS", "Oceania", 3300), ("NZL", "Oceania", 2400), ("SAU", "Asia", 1000), ("TUR", "Asia", 500),
    ]
    generic = [
        "Mr. President, I congratulate you on your election to the presidency of this session.",
        "My delegation reaffirms its commitment to the principles of the Charter of the United Nations.",
        "The international community must act with unity and resolve.",
        "We welcome the report of the Secretary-General and support his efforts.",
        "Our country remains a firm believer in multilateralism and dialogue.",
        "The reform of the Security Council remains a priority for my Government.",
        "We call on all parties to respect international law and human rights.",
        "Peace and security are the foundation of prosperity for all nations.",
        "I thank the Assembly for its attention.",
        "The situation in the Middle East continues to demand our full attention.",
        "Nuclear disarmament must remain at the top of the agenda of this Organization.",
        "My Government attaches great importance to the work of the specialized agencies.",
    ]
    topical = {
        "trade": ["International trade must be free and fair for all countries.",
                  "Protectionism in the developed world harms the exports of developing nations.",
                  "The multilateral trade system needs to be strengthened.",
                  "Trade barriers and tariffs distort the world economy."],
        "debt": ["The external debt burden is crushing the economies of the developing world.",
                 "We call for debt relief and a restructuring of debt repayments.",
                 "Debt servicing absorbs resources that should go to health and education."],
        "development": ["Economic development is the central challenge of our time.",
                        "Development assistance must be increased to the agreed targets.",
                        "Poverty eradication requires sustained development cooperation."],
        "climate": ["Climate change is a threat to the survival of our planet.",
                    "Greenhouse gas emissions must be reduced substantially.",
                    "We urge all states to implement the climate agreement."],
        "terrorism": ["Terrorism is a threat to all nations and must be defeated.",
                      "We condemn all acts of terrorism wherever they occur."],
    }
    rows = []
    gdp_rows = []
    for iso, region, base in countries:
        growth = rng.normal(0.025, 0.008)
        for year in range(1970, 2019):
            t = year - 1970
            gdp = base * (1 + growth) ** t * np.exp(rng.normal(0, 0.05))
            log_gdp = np.log(gdp)
            gdp_rows.append({"Country Name": iso, "Country Code": iso, "region": region,
                             "year": str(year), "gdp": round(gdp * 3.5, 1)})
            # keyword propensities (per 100 sentences) as functions of income and time
            p = {
                "trade": 2 + 0.6 * (log_gdp - 6) + 0.04 * t,
                "debt": max(0.3, 6 - 1.2 * (log_gdp - 6) + (3 if 1980 <= year <= 1995 else 0)),
                "development": max(0.5, 8 - 1.5 * (log_gdp - 6)),
                "climate": 0.1 + (0.0 if year < 1988 else 0.25 * (year - 1988)),
                "terrorism": 0.3 + (3.0 if year >= 2001 else 0),
            }
            n_sent = int(rng.integers(50, 110))
            sents = []
            for _ in range(n_sent):
                u = rng.random() * 100
                cum = 0
                chosen = None
                for kw, rate in p.items():
                    cum += max(rate, 0)
                    if u < cum:
                        chosen = kw
                        break
                if chosen is None:
                    sents.append(generic[rng.integers(len(generic))])
                else:
                    pool = topical[chosen]
                    sents.append(pool[rng.integers(len(pool))])
            text = " ".join(sents)
            rows.append({"country": iso, "session": year - 1945, "year": year, "text": text})
    df = pd.DataFrame(rows)
    # plant typical data problems
    idx = rng.choice(df.index, 40, replace=False)
    df.loc[idx[:15], "text"] = "\ufeff" + df.loc[idx[:15], "text"]
    df.loc[idx[15:30], "text"] = df.loc[idx[15:30], "text"].str.replace(". ", ".\n\n", regex=False)
    df.loc[idx[30:], "text"] = "   " + df.loc[idx[30:], "text"].str.replace(" ", "   ", regex=False) + "  "
    df = pd.concat([df, df.iloc[[5, 200, 900]]], ignore_index=True)   # duplicates
    df.loc[df.index[-1], "country"] = "chn"                            # inconsistent code

    # Write a fake World Bank download in the real file layout, so Chapter 10 can practice on it
    g = pd.DataFrame(gdp_rows)
    wide = g.pivot(index=["Country Name", "Country Code"], columns="year", values="gdp").reset_index()
    wide.insert(2, "Indicator Name", "GDP per capita (current US$)")
    wide.insert(3, "Indicator Code", "NY.GDP.PCAP.CD")
    wide[""] = np.nan                                                   # the trailing empty column
    with open("data/raw/worldbank_gdp_pc_wide.csv", "w", encoding="utf-8") as f:
        f.write('"Data Source","World Development Indicators",\n\n"Last Updated Date","2026-01-01",\n\n')
        wide.to_csv(f, index=False)
    latest = g[g["year"] == "2018"].set_index("Country Code")["gdp"]
    income = pd.qcut(latest, 4, labels=["Low income", "Lower middle income",
                                        "Upper middle income", "High income"])
    meta = pd.DataFrame({
        "Country Code": latest.index,
        "Region": [dict((c, r) for c, r, _ in countries)[c] for c in latest.index],
        "IncomeGroup": income.values.astype(str),
        "SpecialNotes": "",
        "TableName": latest.index,
    })
    meta.to_csv("data/raw/worldbank_country_metadata.csv", index=False)
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)  # shuffle rows

if speeches is None:
    SYNTHETIC = True
    speeches = make_synthetic_corpus()
    print("Built synthetic corpus:", speeches.shape)
    print("Remember: patterns in this data are artificial.")

### 3.4 First look

Whatever the source, these are the first commands on any new dataset. Read the output, do not just run it.

In [ ]:
speeches.head()

In [ ]:
speeches.info()

In [ ]:
# How many speeches per year? value_counts counts unique values and sorts by frequency.
# sort_index() puts the years in order instead.
speeches["year"].value_counts().sort_index().head(10)

In [ ]:
# How many countries, and which years are covered?
print("countries:", speeches["country"].nunique())
print("years:", speeches["year"].min(), "to", speeches["year"].max())

In [ ]:
# Look at one actual speech. The index label of the first row
first_text = speeches.loc[speeches.index[0], "text"]
print(repr(first_text[:400]))      # repr shows invisible characters like \n and \ufeff

### 3.5 Saving intermediate results

A rule for every project: never edit the raw file. Save processed versions to `data/processed/`.
Parquet is faster and keeps types; CSV works everywhere. We save both ways here so you have seen both.

In [ ]:
speeches.to_csv("data/processed/speeches_loaded.csv", index=False)
try:
    speeches.to_parquet("data/processed/speeches_loaded.parquet", index=False)
    print("Saved CSV and Parquet")
except ImportError:
    print("Saved CSV only (install pyarrow for parquet)")

### Exercises 3

1. How many speeches are there in total, and what is the average number of speeches per year?
2. Which five countries have the most speeches in the dataset? (Hint: `value_counts().head()`)
3. Read the saved CSV back in with `pd.read_csv` and check with `.equals()` or by comparing `.shape`
   whether it matches `speeches`.

<details>
<summary>Solution</summary>

```python
print(len(speeches), speeches.groupby("year").size().mean())
print(speeches["country"].value_counts().head())
reloaded = pd.read_csv("data/processed/speeches_loaded.csv")
print(reloaded.shape == speeches.shape)
```
</details>

In [ ]:
# Your solution to Exercises 3

## Chapter 4: Looking at data: selecting, filtering, sorting

Three ways to pick rows and columns:

- `df[col]` or `df[[col1, col2]]` for columns.
- `df.loc[row_labels, col_labels]` by **label**. Slices include both ends.
- `df.iloc[row_positions, col_positions]` by **position**, like Python lists. Slices exclude the end.

And the one that matters most: **boolean indexing**, `df[condition]`, where `condition` is a Series of True/False.

In [ ]:
# Positional: first three rows, first two columns
speeches.iloc[0:3, 0:2]

In [ ]:
# Label-based: rows with index labels 0 to 2 (inclusive), the named columns
speeches.loc[0:2, ["country", "year"]]

### 4.2 Boolean indexing

Build a condition, inspect it, then pass it into the brackets.

In [ ]:
is_che = speeches["country"] == "CHE"
print(type(is_che))
print(is_che.sum())            # True counts as 1, so sum() counts matching rows
is_che.head()

In [ ]:
che = speeches[is_che]
che[["country", "year", "session"]].sort_values("year").head()

In [ ]:
# Combine conditions with & (and), | (or), ~ (not). Parentheses are required.
mask = (speeches["country"] == "CHE") & (speeches["year"] >= 2000)
speeches.loc[mask, ["country", "year"]].sort_values("year")

In [ ]:
# isin for membership in a list
g7 = ["USA", "CAN", "GBR", "FRA", "DEU", "ITA", "JPN"]
speeches[speeches["country"].isin(g7)]["country"].value_counts()

In [ ]:
# query is a readable alternative for simple conditions
speeches.query("country == 'DEU' and year < 1975")[["country", "year"]]

In [ ]:
# between is handy for ranges
speeches[speeches["year"].between(1989, 1991)]["year"].value_counts().sort_index()

### 4.3 Sorting

`sort_values` sorts by one or more columns. `ascending` can be a list to sort columns in different directions.

In [ ]:
speeches.sort_values(["year", "country"]).head()

In [ ]:
speeches.sort_values(["year", "country"], ascending=[False, True]).head()

### 4.4 Counting and unique values

`value_counts`, `nunique`, `unique` answer "what is in this column?" questions.

In [ ]:
print(speeches["country"].nunique())
print(sorted(speeches["country"].unique())[:12])
speeches["year"].value_counts(normalize=True).sort_index().head()   # shares instead of counts

### 4.5 Chaining

Pandas code is often written as a chain of methods. Wrapping the chain in parentheses lets you put each
step on its own line, which reads like a recipe.

In [ ]:
(
    speeches
    .loc[speeches["year"] >= 2010]
    .sort_values(["year", "country"])
    .loc[:, ["year", "country"]]
    .head(8)
)

### Exercises 4

1. Select all speeches by African countries in your data (pick a few ISO codes) given between 1980 and 1990.
   How many are there?
2. Show the three most recent speeches of the United States with their year and session.
3. Using a chain, select speeches from 1970 to 1975, sort by country, and show the columns `country`, `year`.

<details>
<summary>Solution</summary>

```python
africa = ["NGA", "KEN", "EGY", "ZAF", "GHA", "ETH", "TZA", "MAR"]
m = speeches["country"].isin(africa) & speeches["year"].between(1980, 1990)
print(m.sum())

print(speeches[speeches["country"] == "USA"].sort_values("year", ascending=False)[["year", "session"]].head(3))

(speeches.loc[speeches["year"].between(1970, 1975)]
         .sort_values("country")[["country", "year"]])
```
</details>

In [ ]:
# Your solution to Exercises 4

## Chapter 5: Cleaning

Real data arrives dirty. The cleaning checklist for a text corpus:

1. Types: are numbers numbers, text text?
2. Missing values: where, how many, what to do?
3. Duplicates.
4. Inconsistent categories (e.g. `chn` vs `CHN`).
5. Text problems: invisible characters, line breaks, repeated whitespace, encoding leftovers.

We take them in order. The output of this chapter is a cleaned `speeches` DataFrame.

In [ ]:
# 1. Types. If 'year' came in as text, convert it with astype or pd.to_numeric.
speeches["year"] = pd.to_numeric(speeches["year"], errors="coerce").astype("Int64")
speeches["session"] = pd.to_numeric(speeches["session"], errors="coerce").astype("Int64")
speeches.dtypes

`errors="coerce"` turns anything that cannot be converted into `NaN` (missing) instead of raising an error.
`Int64` (capital I) is an integer type that allows missing values; plain `int64` does not.

In [ ]:
# 2. Missing values. isna() gives True where a value is missing.
print(speeches.isna().sum())
print("share of rows with any missing value:", speeches.isna().any(axis=1).mean())

Decide per column. For `text`, a missing value means the row is useless: drop it. For a numeric variable in
other projects you might fill (`fillna`) or keep as missing. Document the decision in a comment.

In [ ]:
before = len(speeches)
speeches = speeches.dropna(subset=["text", "country", "year"])
print("dropped", before - len(speeches), "rows with missing text/country/year")

In [ ]:
# 3. Duplicates. First inspect, then drop.
dup_mask = speeches.duplicated(subset=["country", "year"], keep=False)
print("rows involved in a duplicate country-year:", dup_mask.sum())
speeches[dup_mask].sort_values(["country", "year"])[["country", "year", "session"]].head(10)

Wait. Before dropping, look at what the duplicates are. In the real corpus there are a handful of cases
with two speeches for the same country and year (for example a head of state and a foreign minister).
Dropping blindly loses data; keeping both breaks the "one row per country-year" assumption used in the
merge later. A defensible rule: keep the longer speech. We implement that rule now.

In [ ]:
speeches["text_len"] = speeches["text"].str.len()
speeches = (
    speeches
    .sort_values("text_len", ascending=False)
    .drop_duplicates(subset=["country", "year"], keep="first")
    .drop(columns="text_len")
)
print(len(speeches), "rows after keeping the longest speech per country-year")

In [ ]:
# 4. Inconsistent categories. Upper-case the country code and strip whitespace.
print("before:", speeches["country"].str.isupper().all())
speeches["country"] = speeches["country"].str.strip().str.upper()
print("after:", speeches["country"].str.isupper().all())

# Run the duplicate check again: a code that only differed in case may now collide
print("duplicates after upper-casing:", speeches.duplicated(subset=["country", "year"]).sum())

In [ ]:
# If upper-casing created duplicates, resolve them with the same rule as before
speeches["text_len"] = speeches["text"].str.len()
speeches = (
    speeches.sort_values("text_len", ascending=False)
            .drop_duplicates(subset=["country", "year"], keep="first")
            .drop(columns="text_len")
)
len(speeches)

### 5.2 The `.str` accessor

Every string method from Chapter 1 is available for a whole column through `.str`. The methods return a
new Series; assign it back to keep the result. Several `.str` calls can be chained.

In [ ]:
# Look for the problems first. repr shows invisible characters.
sample = speeches["text"].iloc[:3]
for t in sample:
    print(repr(t[:120]))

In [ ]:
# How many speeches contain a byte order mark, a line break, or doubled spaces?
print("BOM:",         speeches["text"].str.contains("\ufeff").sum())
print("line breaks:", speeches["text"].str.contains("\n").sum())
print("double spaces:", speeches["text"].str.contains("  ").sum())

In [ ]:
# Clean in one chain. regex=True lets \s+ match any run of whitespace including newlines.
speeches["text"] = (
    speeches["text"]
    .str.replace("\ufeff", "", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
print("BOM:", speeches["text"].str.contains("\ufeff").sum(),
      " line breaks:", speeches["text"].str.contains("\n").sum(),
      " double spaces:", speeches["text"].str.contains("  ").sum())

### 5.3 A short regex primer

Regular expressions describe text patterns. You need about ten symbols:

| pattern | matches |
|---------|---------|
| `\s`    | any whitespace (space, tab, newline) |
| `\d`    | a digit |
| `\w`    | a word character (letter, digit, underscore) |
| `\b`    | a word boundary (start or end of a word) |
| `.`     | any character |
| `+`     | one or more of the previous thing |
| `*`     | zero or more |
| `?`     | zero or one |
| `[abc]` | one of a, b, c |
| `(x|y)` | x or y |
| `^`, `$` | start, end of the string |

Why `\b` matters for keyword counting: `"trade"` without boundaries also matches "traded" and "trademark".
`r"\btrade\b"` matches only the whole word. Use `r"\btrad(e|es|ed|ing)\b"` to capture inflections on purpose.

In [ ]:
t = "Trade talks: traders traded trademarks. Free trade!"
print(len(re.findall("trade", t.lower())))
print(len(re.findall(r"\btrade\b", t.lower())))
print(len(re.findall(r"\btrad(e|es|ed|ers?|ing)\b", t.lower())))

### 5.4 Keep a lowercase copy

Most text measures are case-insensitive. Keeping a lowercase column avoids calling `.str.lower()`
over and over on long texts, which is slow.

In [ ]:
speeches["text_lower"] = speeches["text"].str.lower()
speeches = speeches.sort_values(["year", "country"]).reset_index(drop=True)
speeches.head(3)

### 5.5 Historical country codes

The real corpus contains codes for states that no longer exist (YUG, CSK, DDR, YDYE) and the EU as
an observer. The World Bank data in Chapter 10 has no rows for them, so after the merge they will show
up as missing. That is fine as long as you know why. Keep a note of such codes in your project.

In [ ]:
historical = ["YUG", "CSK", "DDR", "YDYE", "EU", "YMD", "YAR"]
print(speeches[speeches["country"].isin(historical)]["country"].value_counts())

In [ ]:
speeches.to_csv("data/processed/speeches_clean.csv", index=False)
print("saved", speeches.shape)

### Exercises 5

1. Count how many speeches end with the sentence "I thank the Assembly for its attention." (Hint:
   `.str.endswith`, or `.str.contains` with `$`.)
2. Create a column `n_sentences` by counting occurrences of `.`, `!` and `?` with `.str.count` and a regex.
3. Which country has the longest average speech (in characters)? (Hint: `.str.len()`, then `groupby`
   which comes in Chapter 8, or `sort_values` on a per-speech basis for now.)

<details>
<summary>Solution</summary>

```python
print(speeches["text"].str.endswith("I thank the Assembly for its attention.").sum())
speeches["n_sentences"] = speeches["text"].str.count(r"[.!?]")
print(speeches.assign(n_chars=speeches["text"].str.len())
              .groupby("country")["n_chars"].mean().idxmax())
```
</details>

In [ ]:
# Your solution to Exercises 5

## Chapter 6: New columns and basic text measures

Derived variables are the raw material of the analysis. Four ways to create them, from fastest to slowest:

1. Vectorized arithmetic and `.str` methods on whole columns (always try this first).
2. `np.where` and `pd.cut` for conditional and binned variables.
3. `.map` with a dictionary to recode values.
4. `.apply` with a Python function, row by row (slow, last resort).

In [ ]:
# Word count: split on whitespace, count elements
speeches["n_words"] = speeches["text"].str.split().str.len()

# Character count and average word length
speeches["n_chars"] = speeches["text"].str.len()
speeches["avg_word_len"] = speeches["n_chars"] / speeches["n_words"]

# Sentence count
speeches["n_sentences"] = speeches["text"].str.count(r"[.!?]")
speeches["words_per_sentence"] = speeches["n_words"] / speeches["n_sentences"]

speeches[["country", "year", "n_words", "n_sentences", "words_per_sentence", "avg_word_len"]].describe()

In [ ]:
# np.where(condition, value_if_true, value_if_false)
speeches["post_cold_war"] = np.where(speeches["year"] >= 1991, 1, 0)
speeches["post_cold_war"].value_counts()

In [ ]:
# pd.cut bins a numeric variable. Decades are the standard example.
bins = list(range(1970, 2031, 10))                      # 1970, 1980, ..., 2030
labels = [f"{b}s" for b in bins[:-1]]                   # "1970s", "1980s", ...
speeches["decade"] = pd.cut(speeches["year"], bins=bins, labels=labels, right=False)
speeches["decade"].value_counts().sort_index()

In [ ]:
# A simpler decade: integer division. 1987 // 10 * 10 = 1980
speeches["decade_num"] = (speeches["year"] // 10 * 10).astype("Int64")
speeches[["year", "decade", "decade_num"]].drop_duplicates().head()

In [ ]:
# .map with a dictionary recodes values. Unmatched values become NaN, so check for that.
region_map = {
    "CHE": "Europe", "DEU": "Europe", "FRA": "Europe", "GBR": "Europe", "ITA": "Europe", "SWE": "Europe",
    "POL": "Europe", "GRC": "Europe", "USA": "Americas", "CAN": "Americas", "MEX": "Americas",
    "BRA": "Americas", "ARG": "Americas", "CHL": "Americas", "COL": "Americas", "PER": "Americas",
    "CHN": "Asia", "IND": "Asia", "JPN": "Asia", "KOR": "Asia", "IDN": "Asia", "PAK": "Asia",
    "THA": "Asia", "PHL": "Asia", "SAU": "Asia", "TUR": "Asia", "NGA": "Africa", "KEN": "Africa",
    "EGY": "Africa", "ZAF": "Africa", "GHA": "Africa", "ETH": "Africa", "TZA": "Africa", "MAR": "Africa",
    "AUS": "Oceania", "NZL": "Oceania",
}
speeches["region"] = speeches["country"].map(region_map)
print("unmapped countries:", speeches.loc[speeches["region"].isna(), "country"].nunique())
speeches["region"].value_counts(dropna=False)

In Chapter 10 the region and income group come from the World Bank for all countries, which replaces
this hand-made map. The hand-made version is here to show the mechanics of `.map`.

In [ ]:
# .apply runs a Python function on every element. Use it only when no vectorized alternative exists.
def first_sentence(text):
    return text.split(". ")[0]

speeches["opening"] = speeches["text"].apply(first_sentence)
speeches["opening"].value_counts().head(5)

In [ ]:
# assign creates columns inside a chain and returns a new DataFrame
(
    speeches
    .assign(log_words=lambda d: np.log(d["n_words"]))
    .loc[:, ["country", "year", "n_words", "log_words"]]
    .head(3)
)

`lambda d: ...` is an anonymous one-line function. Inside `assign`, `d` is the DataFrame as it stands at
that point in the chain, which is how you refer to a column that was created one step earlier.

### 6.2 Column housekeeping

Rename, drop, reorder, and select columns by type.

In [ ]:
speeches = speeches.drop(columns=["opening", "decade_num"])
speeches = speeches.rename(columns={"n_chars": "n_characters"})

print(speeches.select_dtypes(include="number").columns.tolist())
print(speeches.columns.tolist())

### Exercises 6

1. Create `long_speech`, equal to 1 if the speech has more words than the median speech in the data.
2. Create `era` with three values: "cold_war" (before 1991), "unipolar" (1991 to 2007), "recent" (2008 onwards).
   Use `pd.cut` or nested `np.where`.
3. Create `exclamations` as the number of `!` characters per 1,000 words.

<details>
<summary>Solution</summary>

```python
speeches["long_speech"] = (speeches["n_words"] > speeches["n_words"].median()).astype(int)
speeches["era"] = pd.cut(speeches["year"], bins=[1969, 1990, 2007, 2100],
                         labels=["cold_war", "unipolar", "recent"])
speeches["exclamations"] = 1000 * speeches["text"].str.count("!") / speeches["n_words"]
print(speeches[["long_speech", "era", "exclamations"]].describe(include="all"))
```
</details>

In [ ]:
# Your solution to Exercises 6

## Chapter 7: Keyword counting: the core text method

The simplest defensible text measure is a dictionary count: how often does a speech use words from a
list, normalized by its length. It is transparent, fast, and easy to explain in a presentation.

Design decisions you must make and document:

- **Which words.** Start narrow, then widen. Read a few speeches to see what vocabulary is actually used.
- **Whole words or stems.** `\btrade\b` vs `\btrad\w*`. Stems catch more but also catch noise ("tradition").
- **Normalization.** Always per 1,000 words (or share of words). Raw counts mostly measure speech length.
- **Binary vs intensity.** "Mentions trade at all" and "mentions trade 15 times" are different variables.

In [ ]:
# A keyword dictionary: topic -> regex. Alternatives inside a group are joined with |.
keywords = {
    "trade":       r"\b(trade|trading|tariffs?|exports?|imports?|protectionism)\b",
    "debt":        r"\b(debts?|indebted|creditors?|debt relief)\b",
    "development": r"\b(development|developing|poverty|underdevelop\w*)\b",
    "climate":     r"\b(climate|global warming|greenhouse|emissions?)\b",
    "terrorism":   r"\b(terroris[mt]s?)\b",
    "peace":       r"\b(peace|peaceful|ceasefire)\b",
}

# Count each topic in every speech, then normalize per 1,000 words
for topic, pattern in keywords.items():
    speeches[f"kw_{topic}"] = speeches["text_lower"].str.count(pattern)
    speeches[f"kw_{topic}_pt"] = 1000 * speeches[f"kw_{topic}"] / speeches["n_words"]

speeches.filter(like="_pt").describe().round(2)

`.filter(like="_pt")` selects columns whose name contains a substring. Convenient once you have many
derived columns following a naming convention. Use a convention from the start: `kw_` for raw counts,
`_pt` for per-thousand.

In [ ]:
# Binary indicators: does the speech mention the topic at all?
for topic in keywords:
    speeches[f"mentions_{topic}"] = (speeches[f"kw_{topic}"] > 0).astype(int)

speeches.filter(like="mentions_").mean().round(3)

In [ ]:
# Which speeches talk most about debt? nlargest is sort_values + head in one call.
speeches.nlargest(5, "kw_debt_pt")[["country", "year", "n_words", "kw_debt", "kw_debt_pt"]]

### 7.2 Validate the measure

A dictionary is only as good as its words. Two quick checks: look at the context of matches, and look at
speeches with a high score to see whether they are really about the topic.

In [ ]:
def show_matches(text, pattern, n=5, width=60):
    """Print up to n matches of pattern in text with surrounding context."""
    for i, m in enumerate(re.finditer(pattern, text)):
        if i >= n:
            break
        start, end = max(0, m.start() - width), min(len(text), m.end() + width)
        print("...", text[start:end].replace("\n", " "), "...")

top = speeches.nlargest(1, "kw_trade_pt").iloc[0]
print(top["country"], int(top["year"]))
show_matches(top["text_lower"], keywords["trade"])

### 7.3 Reading one speech when you need to

Sometimes you need the full text. Jupyter truncates long strings in tables; use `print` or `textwrap`.

In [ ]:
import textwrap
one = speeches[(speeches["country"] == "CHE")].sort_values("year").iloc[-1]
print(one["country"], int(one["year"]), "\n")
print(textwrap.fill(one["text"][:1200], width=100))

### Exercises 7

1. Add a topic "security" with a pattern of your choice and compute its per-thousand column.
2. What share of speeches mention both trade and debt?
3. Find the speech with the highest climate intensity before 1990. Print its first 500 characters.
   Is the dictionary doing what you want?

<details>
<summary>Solution</summary>

```python
keywords["security"] = r"\b(security|military|arms|weapons?|disarmament)\b"
speeches["kw_security"] = speeches["text_lower"].str.count(keywords["security"])
speeches["kw_security_pt"] = 1000 * speeches["kw_security"] / speeches["n_words"]

both = (speeches["kw_trade"] > 0) & (speeches["kw_debt"] > 0)
print(both.mean())

early = speeches[speeches["year"] < 1990].nlargest(1, "kw_climate_pt").iloc[0]
print(early["country"], int(early["year"]), early["kw_climate_pt"])
print(early["text"][:500])
```
</details>

In [ ]:
# Your solution to Exercises 7

## Chapter 8: groupby: from speeches to countries and years

`groupby` is the pandas version of "for each group, compute something". The pattern has three parts:

```
df.groupby(keys)[columns].aggregation()
```

Split the rows by `keys`, apply the aggregation to `columns` in each group, combine the results into
a new DataFrame whose index is the group keys. Almost every summary table in a project is a `groupby`.

In [ ]:
# How many speeches per decade?
speeches.groupby("decade", observed=True).size()

In [ ]:
# Average trade intensity per year: one key, one column, one aggregation. The result is a Series.
trade_by_year = speeches.groupby("year")["kw_trade_pt"].mean()
trade_by_year.head()

In [ ]:
# Several aggregations at once with agg. The result is a DataFrame.
speeches.groupby("decade", observed=True)["kw_trade_pt"].agg(["mean", "median", "std", "count"]).round(2)

In [ ]:
# Different aggregations for different columns, with your own output names
summary = speeches.groupby("country").agg(
    n_speeches=("year", "size"),
    first_year=("year", "min"),
    last_year=("year", "max"),
    avg_words=("n_words", "mean"),
    trade_pt=("kw_trade_pt", "mean"),
    debt_pt=("kw_debt_pt", "mean"),
    climate_pt=("kw_climate_pt", "mean"),
)
summary.sort_values("debt_pt", ascending=False).head(8).round(2)

In [ ]:
# Two keys: the result has a MultiIndex (two levels of row labels)
by_region_decade = speeches.groupby(["region", "decade"], observed=True)["kw_debt_pt"].mean()
by_region_decade.head(10)

In [ ]:
# as_index=False keeps the keys as ordinary columns instead of an index. Usually easier to work with.
speeches.groupby(["region", "decade"], observed=True, as_index=False)["kw_debt_pt"].mean().head()

### 8.2 transform: group statistics back onto every row

`agg` collapses each group to one row. `transform` computes the same statistic but returns a value for
every original row, which lets you demean, standardize, or compute shares within groups. This is exactly
what fixed effects do in econometrics: subtract the group mean.

In [ ]:
# Country-demeaned trade intensity: how much does this speech talk about trade compared with that
# country's own average?
speeches["trade_pt_country_mean"] = speeches.groupby("country")["kw_trade_pt"].transform("mean")
speeches["trade_pt_demeaned"] = speeches["kw_trade_pt"] - speeches["trade_pt_country_mean"]

speeches[["country", "year", "kw_trade_pt", "trade_pt_country_mean", "trade_pt_demeaned"]].head()

In [ ]:
# Rank within group: in which year did each country talk most about debt?
speeches["debt_rank_in_country"] = speeches.groupby("country")["kw_debt_pt"].rank(ascending=False)
speeches[speeches["debt_rank_in_country"] == 1][["country", "year", "kw_debt_pt"]].sort_values("year").head(8)

### 8.3 pivot_table and crosstab

A pivot table is a groupby with two keys shown as a grid: one key on the rows, one on the columns.

In [ ]:
pd.pivot_table(speeches, values="kw_climate_pt", index="region", columns="decade",
               aggfunc="mean", observed=True).round(2)

In [ ]:
# crosstab counts combinations of two categorical variables
pd.crosstab(speeches["region"], speeches["mentions_climate"], normalize="index").round(2)

### 8.4 Iterating over groups

Occasionally you need to do something per group that does not fit an aggregation, such as saving a file
per country or fitting a model per region. `for key, group in df.groupby(...)` gives you each sub-DataFrame.

In [ ]:
for region, grp in speeches.groupby("region"):
    print(f"{region:10s} {len(grp):5d} speeches, {grp['country'].nunique():3d} countries")

### Exercises 8

1. For each decade, what share of speeches mentions terrorism? (Hint: the mean of a 0/1 column is a share.)
2. Build a table with countries as rows and the average per-thousand of all six topics as columns. Sort by
   climate. (Hint: `groupby(...)[list_of_columns].mean()`; build the list with a comprehension over
   `keywords`.)
3. Create a column `words_vs_year_avg`: each speech's word count divided by the average word count of all
   speeches in the same year. Which country-year has the highest value?

<details>
<summary>Solution</summary>

```python
print(speeches.groupby("decade", observed=True)["mentions_terrorism"].mean().round(3))

pt_cols = [f"kw_{t}_pt" for t in keywords]
print(speeches.groupby("country")[pt_cols].mean().sort_values("kw_climate_pt", ascending=False).round(2).head())

speeches["words_vs_year_avg"] = speeches["n_words"] / speeches.groupby("year")["n_words"].transform("mean")
print(speeches.nlargest(1, "words_vs_year_avg")[["country", "year", "words_vs_year_avg"]])
```
</details>

In [ ]:
# Your solution to Exercises 8

## Chapter 9: Reshaping: wide and long

The same data can be stored in two shapes.

- **Wide**: one row per unit, one column per variable or per period. Easy for humans to read.
- **Long**: one row per unit-variable (or unit-period) combination, with a column saying which variable it is.
  Easy for computers to group, filter and plot.

`melt` goes wide to long. `pivot` goes long to wide. You will need both, because the World Bank delivers
data wide (one column per year) and the analysis wants it long (one row per country-year).

In [ ]:
# Our keyword columns are wide: one column per topic. Melt them into long form.
pt_cols = [f"kw_{t}_pt" for t in keywords]

long = speeches.melt(
    id_vars=["country", "year", "region"],   # columns that identify the row and are kept
    value_vars=pt_cols,                      # columns that are stacked into one
    var_name="topic",                        # name of the new column holding the old column names
    value_name="per_thousand",               # name of the new column holding the values
)
long["topic"] = long["topic"].str.replace("kw_", "").str.replace("_pt", "")
long.head(8)

In [ ]:
# Long form makes "for every topic" operations trivial
long.groupby(["topic", "region"])["per_thousand"].mean().unstack("region").round(2)

`unstack` moves one level of a MultiIndex from the rows to the columns. It is `pivot` for indexed data.
`stack` is the reverse.

In [ ]:
# And back to wide with pivot: one row per country-year, one column per topic
wide_again = long.pivot(index=["country", "year"], columns="topic", values="per_thousand")
wide_again.head()

In [ ]:
# pivot returns an index with country and year; reset_index makes them columns again
wide_again.reset_index().head(3)

### 9.2 A small wide-to-long example in the World Bank shape

This is the shape the World Bank CSV has. Practice on a tiny version before the real one.

In [ ]:
wb_like = pd.DataFrame({
    "Country Name": ["Switzerland", "India"],
    "Country Code": ["CHE", "IND"],
    "Indicator Code": ["NY.GDP.PCAP.CD", "NY.GDP.PCAP.CD"],
    "1970": [3700, 112],
    "1971": [4100, 118],
    "1972": [4900, 122],
})
wb_like

In [ ]:
wb_long = (
    wb_like
    .melt(id_vars=["Country Name", "Country Code", "Indicator Code"], var_name="year", value_name="gdp_pc")
    .assign(year=lambda d: d["year"].astype(int))
    .rename(columns={"Country Code": "country"})
    .loc[:, ["country", "year", "gdp_pc"]]
    .sort_values(["country", "year"])
)
wb_long

### Exercises 9

1. Melt the `mentions_*` columns into long form and compute, per topic, the share of speeches that
   mention it.
2. Pivot `long` so that rows are years, columns are topics, and values are the mean per-thousand across all
   countries. Show the first five rows.

<details>
<summary>Solution</summary>

```python
m_cols = [f"mentions_{t}" for t in keywords]
ml = speeches.melt(id_vars=["country", "year"], value_vars=m_cols, var_name="topic", value_name="mentions")
print(ml.groupby("topic")["mentions"].mean().round(3))

print(long.pivot_table(index="year", columns="topic", values="per_thousand", aggfunc="mean").head())
```
</details>

In [ ]:
# Your solution to Exercises 9

## Chapter 10: The economic variable: World Bank data and merge

Now the project becomes economics. We attach GDP per capita (current US dollars, indicator
`NY.GDP.PCAP.CD`) to every speech, plus the World Bank region and income group of each country.

Again three options; only one needs to run. All produce the same two DataFrames:

- `gdp`: columns `country`, `year`, `gdp_pc`
- `economies`: columns `country`, `country_name`, `wb_region`, `income_group`

### 10.1 Option A: download with `wbgapi`

`pip install wbgapi` first. The package talks to the World Bank API and returns a wide DataFrame with one
column per year, named `YR1970`, `YR1971`, and so on. Country codes are ISO3, with a few World Bank-specific
exceptions. Aggregates (World, Euro area, ...) are included and must be removed.

In [ ]:
gdp = None
economies = None

try:
    if SYNTHETIC:
        raise RuntimeError("practice corpus in use, real World Bank data would not match its patterns")
    import wbgapi as wb

    raw = wb.data.DataFrame("NY.GDP.PCAP.CD", time=range(1960, 2025), labels=False)
    gdp = (
        raw.reset_index()
           .rename(columns={"economy": "country"})
           .melt(id_vars="country", var_name="year", value_name="gdp_pc")
           .assign(year=lambda d: d["year"].str.replace("YR", "").astype(int))
           .dropna(subset=["gdp_pc"])
    )

    econ_raw = wb.economy.DataFrame(labels=True)
    # Columns and their exact names can change between package versions. Print them if the next line fails.
    economies = (
        econ_raw[~econ_raw["aggregate"]]
        .reset_index()
        .rename(columns={"id": "country", "name": "country_name",
                         "region": "wb_region", "incomeLevel": "income_group"})
        .loc[:, ["country", "country_name", "wb_region", "income_group"]]
    )
    gdp = gdp[gdp["country"].isin(economies["country"])]
    gdp.to_csv("data/raw/worldbank_gdp_pc_long.csv", index=False)
    economies.to_csv("data/raw/worldbank_economies.csv", index=False)
    print("Downloaded from the World Bank:", gdp.shape, economies.shape)
except ImportError:
    print("wbgapi not installed. Trying the CSV option.")
except Exception as e:
    print("Skipped or failed:", type(e).__name__, e)
    print("Trying the CSV option.")

### 10.2 Option B: the CSV from data.worldbank.org

On the indicator page (search "GDP per capita (current US$)"), click Download, CSV. The zip contains
`API_NY.GDP.PCAP.CD_DS2_en_csv_v2_*.csv` (the data, wide, with four metadata lines at the top) and
`Metadata_Country_*.csv` (region and income group). Put both in `data/raw/` with the names below.

If you have no file yet and are working with the synthetic corpus, Chapter 3 already wrote a file in the
same format, so this cell runs either way.

In [ ]:
WB_DATA_CSV = "data/raw/worldbank_gdp_pc_wide.csv"
WB_META_CSV = "data/raw/worldbank_country_metadata.csv"

if gdp is None and os.path.exists(WB_DATA_CSV):
    wide = pd.read_csv(WB_DATA_CSV, skiprows=4)            # skip the four metadata lines
    wide = wide.loc[:, ~wide.columns.str.startswith("Unnamed")]   # drop the trailing empty column
    year_cols = [c for c in wide.columns if c.isdigit()]
    gdp = (
        wide.melt(id_vars=["Country Name", "Country Code"], value_vars=year_cols,
                  var_name="year", value_name="gdp_pc")
            .rename(columns={"Country Code": "country", "Country Name": "country_name"})
            .assign(year=lambda d: d["year"].astype(int))
            .dropna(subset=["gdp_pc"])
            .loc[:, ["country", "year", "gdp_pc"]]
    )
    meta = pd.read_csv(WB_META_CSV)
    economies = (
        meta.rename(columns={"Country Code": "country", "TableName": "country_name",
                             "Region": "wb_region", "IncomeGroup": "income_group"})
            .dropna(subset=["wb_region"])                    # aggregates have no region
            .loc[:, ["country", "country_name", "wb_region", "income_group"]]
    )
    gdp = gdp[gdp["country"].isin(economies["country"])]
    print("Loaded World Bank CSVs:", gdp.shape, economies.shape)

In [ ]:
assert gdp is not None, "No GDP data loaded. Run one of the options above."
print(gdp.head())
print(economies.head())
print(gdp.describe().round(0))

### 10.3 Before merging: check the keys

A merge joins two tables on key columns. Before merging, check that the keys match in type and format,
and know how many rows you expect. Mismatched types (`int` vs `str` year) are the number one cause of
merges that silently produce all-missing results.

In [ ]:
print(speeches[["country", "year"]].dtypes)
print(gdp[["country", "year"]].dtypes)

# Make the key types identical
speeches["year"] = speeches["year"].astype(int)
gdp["year"] = gdp["year"].astype(int)
gdp["country"] = gdp["country"].str.upper().str.strip()

# Is the key unique in the right-hand table? If not, the merge multiplies rows.
print("gdp key unique:", not gdp.duplicated(subset=["country", "year"]).any())
print("speeches key unique:", not speeches.duplicated(subset=["country", "year"]).any())

### 10.4 merge

```
pd.merge(left, right, on=keys, how=...)
```

- `how="inner"`: keep only rows whose key appears in both tables.
- `how="left"`: keep all rows of the left table; right-hand columns are missing where no match exists.
- `how="outer"`: keep everything from both.

For "attach a variable to my observations", use `left` with your main table on the left, then inspect
what did not match. `indicator=True` adds a `_merge` column telling you where each row came from.

In [ ]:
merged = pd.merge(speeches, gdp, on=["country", "year"], how="left", indicator=True)

print(merged["_merge"].value_counts())
print("rows before:", len(speeches), " rows after:", len(merged))

In [ ]:
# Which countries did not match at all? These are usually historical codes or code differences.
unmatched = (
    merged[merged["_merge"] == "left_only"]
    .groupby("country")["year"].agg(["min", "max", "size"])
    .sort_values("size", ascending=False)
)
unmatched.head(15)

Study this table in the real data. Some codes differ between the UN corpus and the World Bank (for example
Romania, Zaire/DR Congo, Kosovo, the Yemens). A small manual recoding dictionary applied with
`.replace` before the merge fixes the important ones. Others (East Germany, Yugoslavia) have no World
Bank series and stay missing. Write the decisions down.

In [ ]:
# Example of a recoding step. Fill the dictionary after inspecting 'unmatched' in your real data.
recode = {
    # "ROM": "ROU",   # Romania
    # "ZAR": "COD",   # Zaire -> DR Congo
}
if recode:
    speeches["country"] = speeches["country"].replace(recode)
    merged = pd.merge(speeches, gdp, on=["country", "year"], how="left", indicator=True)
    print(merged["_merge"].value_counts())

In [ ]:
# Attach region and income group, then drop the indicator column
merged = merged.drop(columns="_merge")
merged = pd.merge(merged, economies, on="country", how="left")
merged["log_gdp_pc"] = np.log(merged["gdp_pc"])

print(merged.shape)
merged[["country", "year", "country_name", "wb_region", "income_group", "gdp_pc", "log_gdp_pc"]].head()

In [ ]:
# Coverage check: share of speeches with a GDP value, by decade
merged.groupby("decade", observed=True)["gdp_pc"].apply(lambda s: s.notna().mean()).round(2)

In [ ]:
# From here on, 'df' is the analysis table
df = merged.copy()
df.to_csv("data/processed/speeches_with_gdp.csv", index=False)
try:
    df.to_parquet("data/processed/speeches_with_gdp.parquet", index=False)
except ImportError:
    pass
print("saved analysis table:", df.shape)

### 10.5 A first substantive table

With the economic variable attached, the question "do poorer countries talk more about debt and
development?" is one `groupby` away.

In [ ]:
df.groupby("income_group")[["kw_debt_pt", "kw_development_pt", "kw_trade_pt", "kw_climate_pt"]].mean().round(2)

### Exercises 10

1. Merge a second World Bank indicator of your choice (for example trade as a share of GDP,
   `NE.TRD.GNFS.ZS`) the same way, with `wbgapi` or a CSV. If you are offline, create a fake one with
   `np.random` for the same country-years to practice the mechanics.
2. What happens to the row count if you use `how="inner"` instead of `"left"` in the GDP merge? Why?
3. Compute the average log GDP per capita per region per decade as a pivot table.

<details>
<summary>Solution</summary>

```python
fake = gdp[["country", "year"]].copy()
fake["trade_share"] = np.random.default_rng(1).uniform(20, 120, len(fake))
df2 = pd.merge(df, fake, on=["country", "year"], how="left")
print(df2["trade_share"].notna().mean())

inner = pd.merge(speeches, gdp, on=["country", "year"], how="inner")
print(len(speeches), len(inner))   # inner drops speeches without a GDP value

print(df.pivot_table(index="wb_region", columns="decade", values="log_gdp_pc", aggfunc="mean", observed=True).round(2))
```
</details>

In [ ]:
# Your solution to Exercises 10

## Chapter 11: Time series tools

Our time unit is the year, stored as an integer. That is enough for most of what we need. The tools in
this chapter all answer "how does something move over time within a unit": lags, differences, growth
rates, rolling means. The key habit: when the data has several countries, every time operation must be
done **within country**, which means `groupby("country")` first and `sort_values("year")` before that.

In [ ]:
df = df.sort_values(["country", "year"]).reset_index(drop=True)

# shift(1) gives the previous row's value within each country: the lag
df["gdp_pc_lag"] = df.groupby("country")["gdp_pc"].shift(1)

# diff and pct_change within country
df["gdp_pc_change"] = df.groupby("country")["gdp_pc"].diff()
df["gdp_growth"] = df.groupby("country")["gdp_pc"].pct_change()

df[["country", "year", "gdp_pc", "gdp_pc_lag", "gdp_pc_change", "gdp_growth"]].head(6).round(3)

Careful: `pct_change` on consecutive rows is only a yearly growth rate if the rows are consecutive
years. If a country is missing a year, the row below is two years later. Check gaps with `diff` on `year`.

In [ ]:
df["year_gap"] = df.groupby("country")["year"].diff()
print(df["year_gap"].value_counts(dropna=False).head())

In [ ]:
# Rolling mean smooths noisy annual series. window=5, centered, within country.
df["trade_pt_smooth"] = (
    df.groupby("country")["kw_trade_pt"]
      .transform(lambda s: s.rolling(window=5, center=True, min_periods=3).mean())
)
df[df["country"] == "CHE"][["year", "kw_trade_pt", "trade_pt_smooth"]].head(8).round(2)

In [ ]:
# Cumulative sum: total number of climate mentions a country has made up to each year
df["climate_cum"] = df.groupby("country")["kw_climate"].cumsum()
df[df["country"] == "IND"][["year", "kw_climate", "climate_cum"]].tail(5)

### 11.2 Real dates and `resample`

When the time variable is a real date (presidential speeches have one), convert it with `pd.to_datetime`,
set it as index, and use `resample` to aggregate to months, quarters or years. A short example on made-up
data, so you have seen the pattern.

In [ ]:
dates = pd.DataFrame({
    "date": ["2021-01-20", "2021-03-11", "2021-04-28", "2021-09-09", "2022-03-01", "2022-10-05"],
    "n_words": [2500, 4100, 6200, 3300, 7100, 2900],
})
dates["date"] = pd.to_datetime(dates["date"])
dates = dates.set_index("date")

print(dates.index.year, dates.index.month_name())
# "YE" = year end, "QE" = quarter end, "ME" = month end (pandas 2.2 and newer; on older versions use "Y", "Q", "M")
print(dates.resample("YE")["n_words"].sum())
print(dates.resample("QE")["n_words"].count())

### Exercises 11

1. Compute, within country, the change in climate intensity compared with five years earlier (`shift(5)`).
   Which country-year has the largest increase?
2. Compute a 10-year rolling average of `kw_debt_pt` within country and show it for one country.
3. For each country, find the first year in which it mentioned climate at all. (Hint: filter to
   `mentions_climate == 1`, then `groupby("country")["year"].min()`.)

<details>
<summary>Solution</summary>

```python
df["climate_change_5y"] = df["kw_climate_pt"] - df.groupby("country")["kw_climate_pt"].shift(5)
print(df.nlargest(1, "climate_change_5y")[["country", "year", "climate_change_5y"]])

df["debt_pt_roll10"] = df.groupby("country")["kw_debt_pt"].transform(lambda s: s.rolling(10, min_periods=5).mean())
print(df[df["country"] == "BRA"][["year", "debt_pt_roll10"]].dropna().head())

first_climate = df[df["mentions_climate"] == 1].groupby("country")["year"].min().sort_values()
print(first_climate.head())
```
</details>

In [ ]:
# Your solution to Exercises 11

## Chapter 12: Plots

Two ways to plot: the `.plot` method on a Series or DataFrame (quick), and matplotlib directly (control).
Both end up as matplotlib figures. The pattern that scales to every situation is:

```python
fig, ax = plt.subplots(figsize=(w, h))   # create a figure and one axes
...plot onto ax...
ax.set_xlabel(...), ax.set_ylabel(...), ax.set_title(...)
fig.savefig("figures/name.png", dpi=150, bbox_inches="tight")
```

Every plot you put in a presentation needs axis labels with units and a title that states the finding.

In [ ]:
# A Series indexed by year plots as a line automatically
by_year = df.groupby("year")[["kw_trade_pt", "kw_debt_pt", "kw_climate_pt", "kw_terrorism_pt"]].mean()

fig, ax = plt.subplots(figsize=(9, 4.5))
by_year.plot(ax=ax)
ax.set_xlabel("Year")
ax.set_ylabel("Mentions per 1,000 words (mean across countries)")
ax.set_title("Topic intensity in UN General Debate speeches")
ax.legend(["trade", "debt", "climate", "terrorism"])
fig.savefig("figures/topics_over_time.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Bar chart from a groupby
fig, ax = plt.subplots(figsize=(7, 4))
(df.groupby("income_group")["kw_debt_pt"].mean()
   .sort_values()
   .plot.barh(ax=ax))
ax.set_xlabel("Debt mentions per 1,000 words")
ax.set_ylabel("")
ax.set_title("Poorer countries talk more about debt")
plt.show()

In [ ]:
# Scatter: GDP per capita against trade intensity, one point per speech
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df["log_gdp_pc"], df["kw_trade_pt"], s=8, alpha=0.3)
ax.set_xlabel("log GDP per capita")
ax.set_ylabel("Trade mentions per 1,000 words")
ax.set_title("Trade intensity and income, all speeches")
plt.show()

### 12.2 Small multiples

One panel per group is often clearer than many lines on one axes. `plt.subplots(nrows, ncols)` returns
an array of axes; loop over groups and axes together.

In [ ]:
regions = [r for r in df["wb_region"].dropna().unique()]
n = len(regions)
ncols = 3
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows), sharex=True, sharey=True)
axes = axes.flatten()

for ax, region in zip(axes, regions):
    sub = df[df["wb_region"] == region].groupby("year")["kw_climate_pt"].mean()
    ax.plot(sub.index, sub.values)
    ax.set_title(region, fontsize=10)
for ax in axes[n:]:
    ax.set_visible(False)            # hide unused panels

fig.suptitle("Climate mentions per 1,000 words by region")
fig.supxlabel("Year")
fig.tight_layout()
plt.show()

In [ ]:
# Histogram and boxplot for distributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
df["n_words"].plot.hist(bins=40, ax=ax1)
ax1.set_xlabel("Words per speech")
df.boxplot(column="kw_development_pt", by="income_group", ax=ax2, rot=30)
ax2.set_title("Development mentions by income group")
ax2.set_xlabel("")
fig.suptitle("")
plt.show()

### Exercises 12

1. Plot the number of speeches per year as a line. Add axis labels.
2. Plot trade intensity over time for three countries of your choice on one axes, with a legend.
3. Make a 2 by 2 grid of scatter plots, one per decade of your choice, of log GDP per capita against
   development intensity.

<details>
<summary>Solution</summary>

```python
fig, ax = plt.subplots(figsize=(8, 3.5))
df.groupby("year").size().plot(ax=ax)
ax.set_xlabel("Year"); ax.set_ylabel("Speeches")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
for c in ["CHE", "IND", "BRA"]:
    sub = df[df["country"] == c]
    ax.plot(sub["year"], sub["kw_trade_pt"], label=c)
ax.legend(); ax.set_xlabel("Year"); ax.set_ylabel("Trade per 1,000 words")
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(9, 7), sharex=True, sharey=True)
for ax, dec in zip(axes.flatten(), ["1970s", "1980s", "1990s", "2000s"]):
    sub = df[df["decade"] == dec]
    ax.scatter(sub["log_gdp_pc"], sub["kw_development_pt"], s=8, alpha=0.4)
    ax.set_title(dec)
fig.tight_layout(); plt.show()
```
</details>

In [ ]:
# Your solution to Exercises 12

## Chapter 13: First statistics

Pandas handles descriptive statistics and correlations. For regressions use `statsmodels`, whose output
looks like what you know from R. The question here: is topic intensity related to income, holding time
and country constant? This is a panel: countries observed over years. The natural specification is a
two-way fixed effects regression, which is exactly what the `transform` demeaning in Chapter 8 does by hand.

In [ ]:
# Correlation matrix of the main variables
cols = ["log_gdp_pc", "kw_trade_pt", "kw_debt_pt", "kw_development_pt", "kw_climate_pt", "n_words"]
df[cols].corr().round(2)

In [ ]:
# Correlation within a group: by decade
df.groupby("decade", observed=True)[["log_gdp_pc", "kw_debt_pt"]].corr().round(2).unstack().iloc[:, 1]

In [ ]:
import statsmodels.formula.api as smf

reg = df.dropna(subset=["log_gdp_pc", "kw_debt_pt"])

# Pooled OLS: formula syntax like R
m1 = smf.ols("kw_debt_pt ~ log_gdp_pc", data=reg).fit()
print(m1.summary().tables[1])

In [ ]:
# Add year fixed effects. C(...) treats the variable as categorical, which creates one dummy per year.
m2 = smf.ols("kw_debt_pt ~ log_gdp_pc + C(year)", data=reg).fit()
print("coefficient on log GDP:", round(m2.params["log_gdp_pc"], 3),
      " se:", round(m2.bse["log_gdp_pc"], 3), " n:", int(m2.nobs))

In [ ]:
# Two-way fixed effects with standard errors clustered by country
m3 = smf.ols("kw_debt_pt ~ log_gdp_pc + C(year) + C(country)", data=reg).fit(
    cov_type="cluster", cov_kwds={"groups": reg["country"]}
)
print("coefficient on log GDP:", round(m3.params["log_gdp_pc"], 3),
      " clustered se:", round(m3.bse["log_gdp_pc"], 3))

Interpretation discipline. Within a country over time, a higher log GDP per capita is associated with
fewer (or more) debt mentions per 1,000 words. This is a correlation conditional on fixed effects, not an
effect of income on speech. Income and the topics leaders choose are jointly determined by crises, politics
and world events. Say exactly that in your write-up.

In [ ]:
# Put several models in one table. A small helper that you can reuse.
def coef_table(models, var):
    rows = []
    for name, m in models.items():
        rows.append({"model": name, "coef": m.params[var], "se": m.bse[var],
                     "p": m.pvalues[var], "n": int(m.nobs), "r2": m.rsquared})
    return pd.DataFrame(rows).set_index("model").round(3)

coef_table({"pooled": m1, "year FE": m2, "two-way FE, clustered": m3}, "log_gdp_pc")

In [ ]:
# Loop a regression over several outcomes with the long-form table from Chapter 9
results = []
for topic in keywords:
    y = f"kw_{topic}_pt"
    sub = df.dropna(subset=["log_gdp_pc", y])
    m = smf.ols(f"{y} ~ log_gdp_pc + C(year) + C(country)", data=sub).fit(
        cov_type="cluster", cov_kwds={"groups": sub["country"]})
    results.append({"topic": topic, "coef": m.params["log_gdp_pc"], "se": m.bse["log_gdp_pc"],
                    "p": m.pvalues["log_gdp_pc"]})
pd.DataFrame(results).set_index("topic").round(3)

### Exercises 13

1. Regress climate intensity on log GDP per capita with year fixed effects only. What does the sign say,
   and why might year fixed effects matter a lot for climate specifically?
2. Add `n_words` as a control to one of the regressions. Does the coefficient on log GDP move?
3. Run the two-way FE regression for debt only on the 1980s and 1990s (the debt crisis years). Compare
   the coefficient with the full-sample one.

<details>
<summary>Solution</summary>

```python
mc = smf.ols("kw_climate_pt ~ log_gdp_pc + C(year)", data=reg).fit()
print(mc.params["log_gdp_pc"])   # year FE absorb the global rise of the climate topic

m2b = smf.ols("kw_debt_pt ~ log_gdp_pc + n_words + C(year)", data=reg).fit()
print(m2.params["log_gdp_pc"], m2b.params["log_gdp_pc"])

sub = reg[reg["year"].between(1980, 1999)]
m3b = smf.ols("kw_debt_pt ~ log_gdp_pc + C(year) + C(country)", data=sub).fit()
print(m3.params["log_gdp_pc"], m3b.params["log_gdp_pc"])
```
</details>

In [ ]:
# Your solution to Exercises 13

## Chapter 14: Word frequencies and distinctive words

Dictionary counts need you to choose the words. The reverse approach lets the data suggest them: split
speeches into words, count, and compare groups. This chapter uses the split/explode/value_counts pattern,
then TF-IDF from scikit-learn to find words that distinguish one group from the rest.

In [ ]:
# Tokenize: lowercase, keep only letters, split. str.findall with a regex is a robust tokenizer.
tokens = df["text_lower"].str.findall(r"\b[a-z]{3,}\b")     # words with 3+ letters
tokens.head(2)

In [ ]:
# explode turns a column of lists into one row per element, repeating the other columns
words = (
    df[["country", "year", "decade", "income_group"]]
    .assign(word=tokens)
    .explode("word")
)
print(words.shape)
words.head()

In [ ]:
# Stopwords: frequent function words that carry no topic. Extend this list as you see fit.
stopwords = set("""
the of and to in that is for we our with this on by be are it as has have from which at not will
all its an or was their these those been can they also must should would there but one more than
his her he she who what when where how into such only other some them very about between through
its my your a i you if so no yes do does did had were being above under over after before
""".split())

words = words[~words["word"].isin(stopwords)]
words["word"].value_counts().head(20)

In [ ]:
# Most frequent words per decade: groupby + value_counts + a trick to keep the top n per group
top_by_decade = (
    words.groupby("decade", observed=True)["word"].value_counts()
         .groupby(level=0, observed=True).head(8)
)
top_by_decade

In [ ]:
# Relative frequency: how much more common is a word in one income group than overall?
freq = words.groupby("income_group")["word"].value_counts(normalize=True).rename("share").reset_index()
overall = words["word"].value_counts(normalize=True).rename("overall")
freq = freq.merge(overall, left_on="word", right_index=True)
freq["ratio"] = freq["share"] / freq["overall"]

(freq[freq["share"] > 0.002]
     .sort_values("ratio", ascending=False)
     .groupby("income_group", observed=True).head(5)
     .round(4))

### 14.2 TF-IDF: distinctive words per group

TF-IDF weights a word by how often it appears in a document (TF) and how rare it is across documents
(IDF), so common-everywhere words get a low score. We treat each decade as one big document and ask
which words characterize it.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

docs = df.groupby("decade", observed=True)["text_lower"].apply(" ".join)

vec = TfidfVectorizer(stop_words="english", min_df=1, max_features=5000, token_pattern=r"\b[a-z]{3,}\b")
X = vec.fit_transform(docs)

tfidf = pd.DataFrame(X.toarray(), index=docs.index, columns=vec.get_feature_names_out())
for dec in tfidf.index:
    top = tfidf.loc[dec].sort_values(ascending=False).head(6)
    print(dec, ":", ", ".join(top.index))

On the practice corpus these lists are almost identical across decades, because every decade draws from
the same small pool of sentences. On the real corpus they differ a lot and are worth reading closely.

Use these lists to improve the keyword dictionary in Chapter 7. That loop, data-driven suggestion
then transparent dictionary, is a good workflow for a seminar project: you can explain every word you count.

### Exercises 14

1. Find the ten most frequent words in speeches by high-income countries after 2010.
2. Build the TF-IDF table with regions as documents and print the top words per region.
3. Count how many distinct words (vocabulary size) each country uses on average per speech.
   (Hint: `tokens.apply(lambda t: len(set(t)))`, then groupby.)

<details>
<summary>Solution</summary>

```python
w = words[(words["income_group"] == "High income") & (words["year"] > 2010)]
print(w["word"].value_counts().head(10))

docs_r = df.dropna(subset=["wb_region"]).groupby("wb_region")["text_lower"].apply(" ".join)
Xr = vec.fit_transform(docs_r)
tr = pd.DataFrame(Xr.toarray(), index=docs_r.index, columns=vec.get_feature_names_out())
for r in tr.index:
    print(r, ":", ", ".join(tr.loc[r].sort_values(ascending=False).head(6).index))

df["vocab_size"] = tokens.apply(lambda t: len(set(t)))
print(df.groupby("country")["vocab_size"].mean().sort_values(ascending=False).head())
```
</details>

In [ ]:
# Your solution to Exercises 14

## Chapter 15: From notebook to project

Everything above is exploration. A project needs a pipeline you can rerun from scratch, a question,
and outputs. This chapter shows the structure.

### 15.1 One function per stage

Each stage takes a DataFrame and returns a DataFrame. No global variables inside. Then the whole pipeline
is three lines, and you can rerun it after changing the keyword dictionary.

In [ ]:
def load_speeches(kaggle_csv="data/raw/un-general-debates.csv", txt_dir="data/raw/ungdc"):
    if os.path.exists(kaggle_csv):
        return pd.read_csv(kaggle_csv)
    if os.path.isdir(txt_dir):
        return load_from_txt_files(txt_dir)
    return make_synthetic_corpus()


def clean_speeches(raw):
    out = raw.copy()
    out["year"] = pd.to_numeric(out["year"], errors="coerce")
    out = out.dropna(subset=["text", "country", "year"])
    out["year"] = out["year"].astype(int)
    out["country"] = out["country"].str.strip().str.upper()
    out["text"] = (out["text"].str.replace("\ufeff", "", regex=False)
                              .str.replace(r"\s+", " ", regex=True).str.strip())
    out["text_len"] = out["text"].str.len()
    out = (out.sort_values("text_len", ascending=False)
              .drop_duplicates(subset=["country", "year"]).drop(columns="text_len"))
    out["text_lower"] = out["text"].str.lower()
    out["n_words"] = out["text"].str.split().str.len()
    out["decade"] = (out["year"] // 10 * 10).astype(str) + "s"
    return out.sort_values(["country", "year"]).reset_index(drop=True)


def add_keywords(clean, keyword_dict):
    out = clean.copy()
    for topic, pattern in keyword_dict.items():
        out[f"kw_{topic}"] = out["text_lower"].str.count(pattern)
        out[f"kw_{topic}_pt"] = 1000 * out[f"kw_{topic}"] / out["n_words"]
        out[f"mentions_{topic}"] = (out[f"kw_{topic}"] > 0).astype(int)
    return out


def add_economics(kw, gdp_table, economies_table):
    out = pd.merge(kw, gdp_table, on=["country", "year"], how="left")
    out = pd.merge(out, economies_table, on="country", how="left")
    out["log_gdp_pc"] = np.log(out["gdp_pc"])
    return out


pipeline_df = add_economics(add_keywords(clean_speeches(load_speeches()), keywords), gdp, economies)
print(pipeline_df.shape)
try:
    pipeline_df.to_parquet("data/processed/analysis_table.parquet", index=False)
except ImportError:
    pipeline_df.to_csv("data/processed/analysis_table.csv", index=False)

### 15.2 Move the functions into a module

Copy the four functions above into a file `speech_tools.py` next to the notebook. Then the notebook starts
with

```python
from speech_tools import load_speeches, clean_speeches, add_keywords, add_economics
```

and stays short. If you change the module, restart the kernel or run
`%load_ext autoreload` and `%autoreload 2` once at the top of the notebook.

### 15.3 Git

```
git init
echo "data/" >> .gitignore
echo ".ipynb_checkpoints/" >> .gitignore
git add .
git commit -m "Initial pipeline"
```

Data stays out of the repository. Add a `README.md` with the download links and the exact file names the
code expects, so that a reader can reproduce the setup.

### 15.4 Suggested project structure for the seminar

1. Question (one sentence) and why it is economic.
2. Data: corpus description, coverage table (speeches per decade, countries matched to World Bank).
3. Measurement: the keyword dictionary, how it was validated, what it misses.
4. Descriptives: the topics-over-time figure, the income-group table.
5. Panel regression with fixed effects and clustered standard errors, with the interpretation caveat.
6. Robustness: a second dictionary variant, a different normalization, dropping one region.
7. Limitations and a sentence on what you would do with more time.

### 15.5 Questions that work with this pipeline

- Debt talk and the debt crises: does debt intensity track the country's own debt-to-GDP (World Bank
  `GC.DOD.TOTL.GD.ZS` or external debt `DT.DOD.DECT.GN.ZS`)?
- Trade rhetoric and trade openness (`NE.TRD.GNFS.ZS`): do countries that trade more talk more about trade,
  and did that change with the WTO in 1995?
- Climate talk and vulnerability: small island states versus oil exporters (`NY.GDP.PETR.RT.ZS`).
- Growth shocks and rhetoric: do speeches change in the year after a recession (negative `gdp_growth`)?

### 15.6 Checklist before you present

- Kernel restarted and all cells run top to bottom without error.
- Every figure has labeled axes with units and a title that states the finding.
- Every number in the slides can be traced to a cell.
- The keyword dictionary is printed in an appendix.
- The repository contains the notebook, the module, a README and no data.

In [ ]:
print("Done. Variables available: speeches, df (analysis table), long, words, gdp, economies, keywords")